# Lambda 1 and 2 Prediction - MLP

### Import Libraries

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score

### Load Data and Define Variables

In [8]:
df = pd.read_excel('D:\Documentos\ic_victor\data_complete_pce.xlsx')
features = ['R', 'S', 'Time']
targets = ['lambda1', 'lambda2'] 
X = df[features]
y = df[targets]

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\victo\AppData\Local\Temp\ipykernel_11620\2892239116.py:1: SyntaxWarning: invalid escape sequence '\D'
  df = pd.read_excel('D:\Documentos\ic_victor\data_complete_pce.xlsx')


### Scale Data

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Train Neural Network

In [10]:
regr_nn = MLPRegressor(hidden_layer_sizes=(100, 100),
                       activation='relu',
                       solver='adam',
                       max_iter=2000,
                       random_state=42)

regr_nn.fit(X_train_scaled, y_train)
import joblib

# Salvar modelo e scaler
joblib.dump(regr_nn, 'ann_model_lambda.joblib')
joblib.dump(scaler, 'scaler_ann.joblib')


['scaler_ann.joblib']

### Predict

In [11]:
y_pred_nn = regr_nn.predict(X_test_scaled)

score_l1 = r2_score(y_test['lambda1'], y_pred_nn[:, 0])
score_l2 = r2_score(y_test['lambda2'], y_pred_nn[:, 1])

print(f"R² Lambda 1: {score_l1:.4f}")
print(f"R² Lambda 2: {score_l2:.4f}")

R² Lambda 1: 0.9998
R² Lambda 2: 0.9990


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score
import joblib

# =========================
# Carregar base de dados
# =========================
df = pd.read_excel(
    r'D:\Documentos\ic_victor\data_complete_pce.xlsx'
)

# =========================
# Normalização dos alvos (λ / λ(t=0))
# =========================
df['lambda1_norm'] = (
    df['lambda1'] /
    df.groupby(['R', 'S'])['lambda1'].transform('first')
)

df['lambda2_norm'] = (
    df['lambda2'] /
    df.groupby(['R', 'S'])['lambda2'].transform('first')
)

# =========================
# Features e targets
# =========================
features = ['R', 'S', 'Time']
targets = ['lambda1_norm', 'lambda2_norm']

X = df[features]
y = df[targets]

# =========================
# Split treino / teste
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# Escalonamento das entradas
# =========================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =========================
# Definição da ANN
# =========================
regr_nn = MLPRegressor(
    hidden_layer_sizes=(100, 100),
    activation='relu',
    solver='adam',
    max_iter=2000,
    random_state=42
)

# =========================
# Treinamento
# =========================
regr_nn.fit(X_train_scaled, y_train)

# =========================
# Avaliação
# =========================
y_pred = regr_nn.predict(X_test_scaled)

r2_l1 = r2_score(y_test['lambda1_norm'], y_pred[:, 0])
r2_l2 = r2_score(y_test['lambda2_norm'], y_pred[:, 1])

print(f"R² Lambda 1 (normalizado): {r2_l1:.4f}")
print(f"R² Lambda 2 (normalizado): {r2_l2:.4f}")

# =========================
# Salvar modelo e scaler
# =========================
joblib.dump(regr_nn, 'ann_model_lambda_norm.joblib')
joblib.dump(scaler, 'scaler_ann.joblib')


R² Lambda 1 (normalizado): -0.7415
R² Lambda 2 (normalizado): 0.4657


['scaler_ann.joblib']

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score
import joblib

# ============================================================
# 1. Carregar base de dados
# ============================================================
df = pd.read_excel(
    r'D:\Documentos\ic_victor\data_complete_pce.xlsx'
)

# Garantir ordenação temporal
df = df.sort_values(['R', 'S', 'Time']).reset_index(drop=True)

# ============================================================
# 2. Normalização correta dos alvos (λ / λ(t=0))
# ============================================================

# Extrair λ no tempo zero
lambda0 = (
    df[df['Time'] == 0]
    .set_index(['R', 'S'])[['lambda1', 'lambda2']]
)

# Verificação de segurança
if lambda0.empty:
    raise ValueError('Não há registros com Time = 0 para normalização.')

# Mapear λ(t=0) para todas as linhas
df = df.join(lambda0, on=['R', 'S'], rsuffix='_0')

# Normalização
df['lambda1_norm'] = df['lambda1'] / df['lambda1_0']
df['lambda2_norm'] = df['lambda2'] / df['lambda2_0']

# ============================================================
# 3. Remover colunas auxiliares
# ============================================================
df = df.drop(columns=['lambda1_0', 'lambda2_0'])

# ============================================================
# 4. Features e targets
# ============================================================
features = ['R', 'S', 'Time']
targets = ['lambda1_norm', 'lambda2_norm']

X = df[features]
y = df[targets]

# ============================================================
# 5. Split treino / teste
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# ============================================================
# 6. Escalonamento das entradas
# ============================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ============================================================
# 7. Definição e treinamento da ANN
# ============================================================
regr_nn = MLPRegressor(
    hidden_layer_sizes=(100, 100),
    activation='relu',
    solver='adam',
    max_iter=3000,
    random_state=42
)

regr_nn.fit(X_train_scaled, y_train)

# ============================================================
# 8. Avaliação
# ============================================================
y_pred = regr_nn.predict(X_test_scaled)

r2_l1 = r2_score(y_test['lambda1_norm'], y_pred[:, 0])
r2_l2 = r2_score(y_test['lambda2_norm'], y_pred[:, 1])

print(f"R² Lambda 1 (normalizado): {r2_l1:.4f}")
print(f"R² Lambda 2 (normalizado): {r2_l2:.4f}")

# ============================================================
# 9. Salvar modelo e scaler
# ============================================================
joblib.dump(regr_nn, 'ann_model_lambda_norm.joblib')
joblib.dump(scaler, 'scaler_ann.joblib')


R² Lambda 1 (normalizado): -0.9475
R² Lambda 2 (normalizado): 0.3634


['scaler_ann.joblib']

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score
import joblib

# ============================================================
# 1. Carregar dados
# ============================================================
df = pd.read_excel(
    r'D:\Documentos\ic_victor\data_complete_pce.xlsx'
)

df = df.sort_values(['R', 'S', 'Time']).reset_index(drop=True)

# ============================================================
# 2. Criar incrementos temporais Δλ
# ============================================================
df['lambda1_prev'] = df.groupby(['R','S'])['lambda1'].shift(1)
df['lambda2_prev'] = df.groupby(['R','S'])['lambda2'].shift(1)

df['delta_lambda1'] = df['lambda1'] - df['lambda1_prev']
df['delta_lambda2'] = df['lambda2'] - df['lambda2_prev']

# Remover Time = 0 (não tem delta)
df = df.dropna().reset_index(drop=True)

# ============================================================
# 3. Features e targets
# ============================================================
features = ['R', 'S', 'Time', 'lambda1_prev', 'lambda2_prev']
targets = ['delta_lambda1', 'delta_lambda2']

X = df[features]
y = df[targets]

# ============================================================
# 4. Split
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ============================================================
# 5. Escalonamento
# ============================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ============================================================
# 6. ANN
# ============================================================
regr_nn = MLPRegressor(
    hidden_layer_sizes=(80, 80),
    activation='tanh',
    solver='adam',
    max_iter=3000,
    random_state=42
)

regr_nn.fit(X_train_scaled, y_train)

# ============================================================
# 7. Avaliação
# ============================================================
y_pred = regr_nn.predict(X_test_scaled)

r2_l1 = r2_score(y_test['delta_lambda1'], y_pred[:,0])
r2_l2 = r2_score(y_test['delta_lambda2'], y_pred[:,1])

print(f"R² Δλ₁: {r2_l1:.4f}")
print(f"R² Δλ₂: {r2_l2:.4f}")

# ============================================================
# 8. Salvar
# ============================================================
joblib.dump(regr_nn, 'ann_model_delta_lambda.joblib')
joblib.dump(scaler, 'scaler_delta_ann.joblib')


R² Δλ₁: 0.2328
R² Δλ₂: 0.2735


d:\Documentos\ic_victor\myenv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(


['scaler_delta_ann.joblib']